# DATA 245 | Experiment v2: corrected synthetic scenarios

**Why this notebook exists.** In `results/results.csv` (v1) the synthetic windows were drawn *with replacement* from one small pool, and retraining then learned from earlier windows drawn from that same pool. By window 8 most scored rows had already been used for retraining. This notebook re-runs the **identical experiment** (same settings, strategies, models, seeds, shift weights) with design 1B and saves the result to `results/results_v2.csv`. `results/results.csv` is **not touched** and notebook `06_experiment.ipynb` is unchanged.

**Design 1B (approved 2026-11-27).** The training rows are split at random (fixed seed) into three disjoint parts: 60% fit the initial model, 20% form a **scoring pool**, 20% form a **history pool**. Each window the models are scored on is drawn from the scoring pool; the rows a retrain may learn from for that window (`Scenario.labelled_windows`) are an independent draw from the history pool with the same shift weights. Draws are still with replacement inside a window; the age-shift weights are unchanged.

**What this is and is not.** The synthetic scenarios are **controlled synthetic experiments, not chronological replays**. The simulation assumes that the labels arriving for window *i* come from a fresh sample of window *i*'s distribution, not from the same customers that were scored. The real replay is unchanged (real consecutive windows, no separate pools) and must reproduce the v1 rows.

**YES rate.** The age shift can legitimately change the share of "yes" in a window. It is reported below as a descriptive diagnostic, not as a pass/fail condition. The PSI checks (age shift clearly visible, no shift before the change) remain pass/fail.

All stream settings remain **PROPOSED** defaults (see `DECISIONS.md`).

In [1]:
import sys, hashlib, time, warnings
from pathlib import Path
p = Path.cwd().resolve()
root = p if (p / 'src').exists() else p.parent
sys.path.insert(0, str(root))

import numpy as np
import pandas as pd

from src.data import chronological_split, encode_target, load_dataset
from src.drift import ks, psi
from src.scenarios import all_scenarios, gradual, shift_weights, stationary, sudden, synthetic_split
from src.settings import (EXPERIMENT_FEATURE_SET, EXPERIMENT_MODELS, LABEL_DELAY_WINDOWS, N_STREAM_WINDOWS,
                          PERIODIC_EVERY_WINDOWS, PSI_ALERT, SEEDS, STRATEGIES, SYNTHETIC_POOL_FRAC, TRIGGER_GROUPS)
from src.strategies import labelled_history, run_experiment, summarise

warnings.filterwarnings('ignore', category=UserWarning)
V1_PATH, V2_PATH = root / 'results' / 'results.csv', root / 'results' / 'results_v2.csv'
sha = lambda path: hashlib.sha256(path.read_bytes()).hexdigest()
V1_SHA_AT_START = sha(V1_PATH)
print('Project root:', root)
print('results.csv SHA-256 at start:', V1_SHA_AT_START)

Project root: /mnt/c/Users/patel/data245-drift-project
results.csv SHA-256 at start: 2b979e130f12d5c617c1664edbf6c67a9e0ad83455a39ef86d5f4f4eba5f8109


In [2]:
pd.Series({
    'models': ', '.join(EXPERIMENT_MODELS), 'feature set': EXPERIMENT_FEATURE_SET, 'strategies': ', '.join(STRATEGIES),
    'seeds': SEEDS, 'windows per stream': N_STREAM_WINDOWS, 'label delay (windows)': LABEL_DELAY_WINDOWS,
    'periodic every (windows)': PERIODIC_EVERY_WINDOWS, 'drift rule': f'any of {TRIGGER_GROUPS} PSI > {PSI_ALERT}',
    'synthetic split': f'{1 - 2 * SYNTHETIC_POOL_FRAC:.0%} fit / {SYNTHETIC_POOL_FRAC:.0%} scoring pool / {SYNTHETIC_POOL_FRAC:.0%} history pool',
}).to_frame('value')

,value
models,"Logistic Regression, Linear SVM"
feature set,baseline_10
strategies,"static, periodic, drift_triggered"
seeds,"[42, 43, 44]"
windows per stream,8
label delay (windows),1
periodic every (windows),2
drift rule,any of ['customer_history'] PSI > 0.25
synthetic split,60% fit / 20% scoring pool / 20% history pool


## 1. Is the age shift still clearly visible? (PSI: pass/fail) and YES rate (descriptive)

**PSI criteria, declared before the run** (age PSI of each scored window against the initial fitting rows, every seed separately):
- Stationary windows, and sudden/gradual windows before the change: PSI < 0.1 (v1 maximum was 0.026).
- Sudden: every window from the change on has PSI >= 0.25.
- Gradual: PSI rises across the ramp (windows 3 to 7) and windows 5 to 8 are >= 0.25.

The observed and *design-expected* YES rate is shown for every window. The expected rate is what the sampling design implies: the weighted average of the scoring-pool labels under that window's shift weights. The z-score says how many binomial standard deviations (n = 1,030 draws) the observed rate is from it. If the check fails the notebook stops here.

In [3]:
train, _dev, _stream = chronological_split(load_dataset())   # stream rows are never read by this notebook
df = load_dataset()
fit_part, scoring_pool, history_pool = synthetic_split(chronological_split(df)[0])
y_pool = encode_target(scoring_pool).to_numpy()
rows = []
for name, build in (('stationary', stationary), ('sudden', sudden), ('gradual', gradual)):
    for seed in SEEDS:
        sc = build(df, seed=seed)
        for w, (window, strength) in enumerate(zip(sc.windows, sc.strengths), start=1):
            expected = float((shift_weights(scoring_pool, strength) * y_pool).sum())
            observed = float(encode_target(window).mean())
            sd = np.sqrt(expected * (1 - expected) / len(window))
            rows.append({'scenario': name, 'seed': seed, 'window': w, 'change_window0': sc.change_window, 'strength': strength,
                         'age_psi': psi(sc.initial_train['age'], window['age']), 'age_ks': ks(sc.initial_train['age'], window['age'])[0],
                         'yes_observed': observed, 'yes_expected_by_design': expected, 'z': (observed - expected) / sd})
tbl = pd.DataFrame(rows)

print('AGE PSI per window (columns = windows 1..8)')
print(tbl.pivot_table(index=['scenario', 'seed'], columns='window', values='age_psi').round(2).to_string())
print('\nAGE KS statistic per window')
print(tbl.pivot_table(index=['scenario', 'seed'], columns='window', values='age_ks').round(2).to_string())

failures = []
for (name, seed), g in tbl.groupby(['scenario', 'seed']):
    p_ = g.sort_values('window')['age_psi'].to_numpy(); cw = None if g['change_window0'].isna().all() else int(g['change_window0'].iloc[0])
    if name == 'stationary':
        ok = (p_ < 0.1).all()
    elif name == 'sudden':
        ok = (p_[:cw] < 0.1).all() and (p_[cw:] >= PSI_ALERT).all()
    else:
        ok = (p_[:cw] < 0.1).all() and (p_[4:] >= PSI_ALERT).all() and (np.diff(p_[cw:7]) > 0).all()
    print(f'PSI criteria {name:10s} seed {seed}: {"PASS" if ok else "FAIL"}')
    if not ok: failures.append((name, seed))
assert not failures, f'PSI acceptance failed: {failures}'

AGE PSI per window (columns = windows 1..8)
window              1     2     3     4     5     6     7     8
scenario   seed                                                
gradual    42    0.01  0.02  0.05  0.17  0.36  0.57  1.05  1.02
           43    0.01  0.01  0.04  0.18  0.39  0.66  0.92  0.98
           44    0.02  0.01  0.06  0.20  0.31  0.67  0.96  0.88
stationary 42    0.01  0.02  0.00  0.01  0.02  0.01  0.01  0.01
           43    0.01  0.01  0.01  0.01  0.01  0.01  0.01  0.00
           44    0.02  0.01  0.01  0.01  0.02  0.02  0.01  0.01
sudden     42    0.01  0.02  0.00  0.01  1.00  1.01  0.97  1.05
           43    0.01  0.01  0.01  0.01  0.94  0.83  1.07  0.92
           44    0.02  0.01  0.01  0.01  0.87  0.93  0.96  0.96

AGE KS statistic per window
window              1     2     3     4     5     6     7     8
scenario   seed                                                
gradual    42    0.02  0.04  0.10  0.18  0.26  0.33  0.43  0.42
           43    0.04  0.03  0.

In [4]:
print('YES rate (%) observed per window')
print((100 * tbl.pivot_table(index=['scenario', 'seed'], columns='window', values='yes_observed')).round(1).to_string())
print('\nYES rate (%) expected from the sampling design (shift weights x scoring-pool labels)')
print((100 * tbl.pivot_table(index=['scenario', 'seed'], columns='window', values='yes_expected_by_design')).round(1).to_string())
print('\nz-score of observed vs design-expected (binomial SD, n = 1,030)')
print(tbl.pivot_table(index=['scenario', 'seed'], columns='window', values='z').round(1).to_string())
print(f"\nwindows with |z| > 2: {(tbl['z'].abs() > 2).sum()} of {len(tbl)}; |z| > 3: {(tbl['z'].abs() > 3).sum()}")
print('Expected by chance alone at |z| > 2: about', round(0.0455 * len(tbl), 1), 'windows')
named = tbl[((tbl.scenario == 'gradual') & (tbl.seed == 42) & (tbl.window == 3)) | ((tbl.scenario == 'gradual') & (tbl.seed == 44) & (tbl.window == 6))]
display(named.round(3))

YES rate (%) observed per window
window             1    2    3    4    5    6    7    8
scenario   seed                                        
gradual    42    4.7  3.7  6.2  4.7  4.0  5.1  3.5  3.5
           43    5.0  5.2  5.8  4.3  4.4  3.9  3.4  5.0
           44    4.0  4.7  4.6  4.8  4.3  2.8  4.8  4.8
stationary 42    4.7  3.7  4.7  4.0  4.8  4.2  4.8  5.5
           43    5.0  5.2  4.0  4.8  5.1  5.0  5.2  5.8
           44    4.0  4.7  4.6  4.3  5.1  3.9  5.4  5.4
sudden     42    4.7  3.7  4.7  4.0  4.6  4.6  4.8  3.5
           43    5.0  5.2  4.0  4.8  3.4  5.1  4.9  3.4
           44    4.0  4.7  4.6  4.3  4.9  5.1  4.8  4.8

YES rate (%) expected from the sampling design (shift weights x scoring-pool labels)
window             1    2    3    4    5    6    7    8
scenario   seed                                        
gradual    42    4.8  4.8  4.7  4.6  4.5  4.4  4.4  4.4
           43    4.8  4.8  4.7  4.6  4.5  4.4  4.4  4.4
           44    4.8  4.8  4.7  4.6  4.5 

,scenario,seed,window,change_window0,strength,age_psi,age_ks,yes_observed,yes_expected_by_design,z
50,gradual,42,3,2.0,0.2,0.049,0.096,0.062,0.047,2.349
69,gradual,44,6,2.0,0.8,0.671,0.349,0.028,0.044,-2.527


## 2. Run the full corrected experiment

4 scenarios x 3 strategies x 2 models x 3 seeds x 8 windows = 576 rows, exactly as in notebook 06.

In [5]:
start = time.perf_counter()
results = run_experiment(df, seeds=SEEDS)
print(f'Total run time: {time.perf_counter() - start:.0f} s')
results.shape

seed 42  real_replay  Logistic Regression  done in   9.9 s


seed 42  real_replay  Linear SVM           done in   6.8 s


seed 42  stationary   Logistic Regression  done in   3.5 s


seed 42  stationary   Linear SVM           done in   3.3 s


seed 42  sudden       Logistic Regression  done in   4.1 s


seed 42  sudden       Linear SVM           done in   4.0 s


... (15 outputs left out)

seed 44  sudden       Linear SVM           done in   4.1 s


seed 44  gradual      Logistic Regression  done in   4.8 s


seed 44  gradual      Linear SVM           done in   3.9 s
Total run time: 127 s


(576, 39)

## 3. Sanity checks (as in notebook 06) plus the row-overlap check

In [6]:
expected_rows = 4 * len(STRATEGIES) * len(EXPERIMENT_MODELS) * len(SEEDS) * N_STREAM_WINDOWS
assert len(results) == expected_rows, (len(results), expected_rows)
same_stream = results.groupby(['scenario', 'model', 'seed', 'window'])[['n_rows', 'n_positive']].nunique()
assert (same_stream == 1).all().all(), 'strategies did not see identical windows'
first = results[results['window'] == 1].groupby(['scenario', 'model', 'seed'])[['pr_auc', 'threshold']].nunique()
assert (first == 1).all().all(), 'strategies did not start from the same model'
assert not results.query("strategy == 'static'")['retrained'].any()
print('Periodic retrains before windows (1-based):', sorted(results.query("strategy == 'periodic' and retrained")['window'].unique()))
assert results[['pr_auc', 'random_pr_auc', 'f1', 'precision', 'recall']].notna().all().all()

# No scored row is ever in any history (data level; the unit tests also spy on every real fit).
for seed in SEEDS:
    for sc in all_scenarios(df, seed=seed):
        scored = set().union(*[set(w.index) for w in sc.windows])
        assert set(sc.initial_train.index).isdisjoint(scored)
        for i in range(N_STREAM_WINDOWS):
            assert set(labelled_history(sc, i).index).isdisjoint(scored) or sc.kind == 'real'
        if sc.kind == 'synthetic':
            assert sc.labelled_windows is not None
print(f'All checks passed: {len(results)} window rows; no scored row in any synthetic training history.')

Periodic retrains before windows (1-based): [np.int64(3), np.int64(5), np.int64(7)]


All checks passed: 576 window rows; no scored row in any synthetic training history.


## 4. Save v2 (v1 is never overwritten)

In [7]:
assert V2_PATH.name == 'results_v2.csv'
results.to_csv(V2_PATH, index=False)
assert sha(V1_PATH) == V1_SHA_AT_START, 'results.csv changed!'
print('Saved', V2_PATH, '| results.csv unchanged, SHA-256', sha(V1_PATH))

Saved /mnt/c/Users/patel/data245-drift-project/results/results_v2.csv | results.csv unchanged, SHA-256 2b979e130f12d5c617c1664edbf6c67a9e0ad83455a39ef86d5f4f4eba5f8109


## 5. v1 versus v2

### 5a. Real replay: deterministic columns must agree

Timing columns are measured per run and differ between machines, so they are **excluded** from the equality check and reported separately. v1 timings are the values recorded in `results.csv` (measured on another machine); v2 timings are measured in this run. They are never combined.

In [8]:
TIMING = ['retrain_seconds', 'monitor_seconds', 'initial_fit_seconds']
KEYS = ['scenario', 'strategy', 'model', 'seed', 'window']
v1, v2 = pd.read_csv(V1_PATH), pd.read_csv(V2_PATH)
assert list(v1.columns) == list(v2.columns)
r1 = v1[v1.scenario == 'real_replay'].sort_values(KEYS).reset_index(drop=True)
r2 = v2[v2.scenario == 'real_replay'].sort_values(KEYS).reset_index(drop=True)
assert len(r1) == len(r2) == 144 and r1[KEYS].equals(r2[KEYS])
diffs = []
for c in [c for c in v1.columns if c not in TIMING]:
    if pd.api.types.is_numeric_dtype(r1[c]) and not pd.api.types.is_bool_dtype(r1[c]):
        a, b = r1[c].astype(float), r2[c].astype(float)
        same_nan = a.isna() & b.isna()
        d = (a - b).abs().where(~same_nan, 0.0).fillna(np.inf)
        diffs.append((c, 'numeric', float(d.max()), int((d > 0).sum())))
    else:
        mism = int((r1[c].fillna('').astype(str) != r2[c].fillna('').astype(str)).sum())   # empty == empty (NaN != NaN would be a false alarm)
        diffs.append((c, 'exact', 0.0 if mism == 0 else np.nan, mism))
diff_table = pd.DataFrame(diffs, columns=['column', 'compare', 'max_abs_difference', 'rows_that_differ'])
print(f'{len(diff_table)} non-timing columns compared on {len(r1)} real-replay rows')
print('Columns with any difference:', int((diff_table.rows_that_differ > 0).sum()))
print('Largest numeric difference:', diff_table.max_abs_difference.max())
display(diff_table[diff_table.rows_that_differ > 0] if (diff_table.rows_that_differ > 0).any() else 'all non-timing real-replay columns are identical')

36 non-timing columns compared on 144 real-replay rows
Columns with any difference: 0
Largest numeric difference: 0.0


'all non-timing real-replay columns are identical'

In [9]:
timing = pd.DataFrame({
    'v1 recorded in results.csv (other machine)': r1[TIMING].sum(),
    'v2 measured in this run': r2[TIMING].sum()})
timing.index.name = 'sum over the 144 real-replay rows (seconds)'
timing.round(2)

,v1 recorded in results.csv (other machine),v2 measured in this run
sum over the 144 real-replay rows (seconds),,
retrain_seconds,23.12,22.45
monitor_seconds,34.25,24.02
initial_fit_seconds,39.09,35.98


### 5b. Synthetic scenarios: v1 (row reuse) versus v2 (separate pools)

Mean over the 8 windows, then over 3 seeds. v1 is known to be contaminated for retraining strategies; it is shown only for comparison.

In [10]:
s1, s2 = summarise(v1), summarise(v2)
cmp = (s1.merge(s2, on=['scenario', 'strategy', 'model'], suffixes=('_v1', '_v2'))
         .query("scenario != 'real_replay'")[['scenario', 'model', 'strategy', 'mean_pr_auc_v1', 'mean_pr_auc_v2', 'mean_random_pr_auc_v1',
                                              'mean_random_pr_auc_v2', 'retrains_v1', 'retrains_v2', 'alerts_v1', 'alerts_v2']])
order = {s: i for i, s in enumerate(STRATEGIES)}
cmp = cmp.assign(o=cmp.strategy.map(order)).sort_values(['scenario', 'model', 'o']).drop(columns='o')
cmp.round(4).to_string(index=False)

'  scenario               model        strategy  mean_pr_auc_v1  mean_pr_auc_v2  mean_random_pr_auc_v1  mean_random_pr_auc_v2  retrains_v1  retrains_v2  alerts_v1  alerts_v2\n   gradual          Linear SVM          static          0.0649          0.0531                 0.0431                 0.0445          0.0          0.0        4.0        4.0\n   gradual          Linear SVM        periodic          0.0692          0.0538                 0.0431                 0.0445          3.0          3.0        4.0        4.0\n   gradual          Linear SVM drift_triggered          0.0694          0.0529                 0.0431                 0.0445          4.0          4.0        4.0        4.0\n   gradual Logistic Regression          static          0.0650          0.0521                 0.0431                 0.0445          0.0          0.0        4.0        4.0\n   gradual Logistic Regression        periodic          0.0690          0.0538                 0.0431                 0.0445     

In [11]:
# Gain of retraining over static (mean PR-AUC), v1 vs v2, synthetic scenarios
g = cmp.pivot_table(index=['scenario', 'model'], columns='strategy', values=['mean_pr_auc_v1', 'mean_pr_auc_v2'])
out = pd.DataFrame({
    'periodic - static (v1)': g['mean_pr_auc_v1']['periodic'] - g['mean_pr_auc_v1']['static'],
    'periodic - static (v2)': g['mean_pr_auc_v2']['periodic'] - g['mean_pr_auc_v2']['static'],
    'drift - static (v1)': g['mean_pr_auc_v1']['drift_triggered'] - g['mean_pr_auc_v1']['static'],
    'drift - static (v2)': g['mean_pr_auc_v2']['drift_triggered'] - g['mean_pr_auc_v2']['static']})
out.round(4)

periodic - static (v1)  \
scenario   model                                         
gradual    Linear SVM                           0.0043   
           Logistic Regression                  0.0040   
stationary Linear SVM                           0.0021   
           Logistic Regression                  0.0020   
sudden     Linear SVM                           0.0055   
           Logistic Regression                  0.0054   

                                periodic - static (v2)  drift - static (v1)  \
scenario   model                                                              
gradual    Linear SVM                           0.0007               0.0045   
           Logistic Regression                  0.0017               0.0042   
stationary Linear SVM                           0.0009               0.0000   
           Logistic Regression                  0.0010               0.0000   
sudden     Linear SVM                           0.0000               0.0063   
           Logistic Regression                  0.0003               0.0062   

                                drift - static (v2)  
scenario   model                                     
gradual    Linear SVM                       -0.0002  
           Logistic Regression               0.0008  
stationary Linear SVM                        0.0000  
           Logistic Regression               0.0000  
sudden     Linear SVM                        0.0004  
           Logistic Regression               0.0006

### 5c. Alerts, detection delay and false alarms in v2 (read from the static strategy, whose drift reference never changes)

In [12]:
rows = []
for sc_name in ['stationary', 'sudden', 'gradual']:
    s = v2[(v2.scenario == sc_name) & (v2.strategy == 'static') & (v2.model == 'Logistic Regression')]
    for seed, g_ in s.groupby('seed'):
        cw = g_.change_window.iloc[0]
        if pd.isna(cw):
            rows.append({'scenario': sc_name, 'seed': seed, 'change_window': None, 'false_alarms': int(g_.alert.sum()), 'first_alert_at_or_after_change': None, 'detection_delay': None})
        else:
            first = g_[(g_.window >= cw) & g_.alert].window.min()
            rows.append({'scenario': sc_name, 'seed': seed, 'change_window': int(cw), 'false_alarms': int(g_[g_.window < cw].alert.sum()),
                         'first_alert_at_or_after_change': first, 'detection_delay': first - cw})
pd.DataFrame(rows)

,scenario,seed,change_window,false_alarms,first_alert_at_or_after_change,detection_delay
0,stationary,42,NaN,0,NaN,NaN
1,stationary,43,NaN,0,NaN,NaN
2,stationary,44,NaN,0,NaN,NaN
3,sudden,42,5.0,0,5.0,0.0
4,sudden,43,5.0,0,5.0,0.0
5,sudden,44,5.0,0,5.0,0.0
6,gradual,42,3.0,0,5.0,2.0
7,gradual,43,3.0,0,5.0,2.0
8,gradual,44,3.0,0,5.0,2.0
